<a href="https://colab.research.google.com/github/AntonRvn/SlovoDataset/blob/main/Slovo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Загрузка библиотек**

In [ ]:
!pip install -q ijson pyarrow

Импорт библиотек

In [ ]:
from google.colab import drive

import ijson
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from tqdm import tqdm

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split, StratifiedKFold

[Текст ссылки](https://)**Подключение к Google Drive**

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Пути к файлам

In [ ]:
JSON_PATH = Path(
    "/content/drive/MyDrive/slovo_mediapipe.json"
)

ANNOTATIONS_PATH = Path(
    "/content/drive/MyDrive/annotations.csv"
)

Загрузка файла аннотаций

In [ ]:
annotations = pd.read_csv(
    ANNOTATIONS_PATH,
    sep='\t'
)

annotations['attachment_id'] = (
    annotations['attachment_id']
    .astype(str)
)

print(annotations.shape)

annotations.head()

(20400, 9)


,attachment_id,text,user_id,height,width,length,train,begin,end
0,44e8d2a0-7e01-450b-90b0-beb7400d2c1e,Ё,185bd3a81d9d618518d10abebf0d17a8,1920,1080,156.0,True,36,112
1,df5b08f0-41d1-4572-889c-8b893e71069b,А,185bd3a81d9d618518d10abebf0d17a8,1920,1080,150.0,True,36,76
2,17f53df4-c467-4aff-9f48-20687b63d49a,Р,185bd3a81d9d618518d10abebf0d17a8,1920,1080,133.0,True,40,97
3,e3add916-c708-4339-ad98-7e2740be29e9,Е,185bd3a81d9d618518d10abebf0d17a8,1920,1080,144.0,True,43,107
4,bd7272ed-1850-48f1-a2a8-c8fed523dc37,Ч,185bd3a81d9d618518d10abebf0d17a8,1920,1080,96.0,True,20,70


Кодирование классов жестов

In [ ]:
label_encoder = LabelEncoder()

annotations['label'] = label_encoder.fit_transform(annotations['text'])

num_classes = annotations['label'].nunique()

print("Количество классов:", num_classes)

Количество классов: 1001


*Создание* словарей labels и train/test split

In [ ]:
label_map = dict(
    zip(
        annotations['attachment_id'],
        annotations['label']
    )
)

train_map = dict(
    zip(
        annotations['attachment_id'],
        annotations['train']
    )
)

Формирование sequence dataset из JSON

In [ ]:
MAX_LEN = 64

sequences = []
labels = []
attachment_ids_list = []

invalid_hands = set()
invalid_points = 0

print("Начинаю обработку JSON...")

with open(JSON_PATH, 'rb') as f:

    parser = ijson.kvitems(f, '')

    for attachment_id, frames in tqdm(parser):

        attachment_id = str(attachment_id)

        if attachment_id not in label_map:
            continue

        if not isinstance(frames, list):
            continue

        sequence = []

        for frame in frames:

            if not isinstance(frame, dict):
                continue

            frame_vector = []

            for hand_name in ['hand 1', 'hand 2']:

                landmarks = frame.get(hand_name)

                if landmarks is None or not isinstance(landmarks, list):
                    frame_vector.extend([0.0] * 63)
                    continue

                if len(landmarks) != 21:
                    invalid_points += 1

                coords = []

                for point in landmarks[:21]:

                    try:
                        coords.extend([
                            float(point['x']),
                            float(point['y']),
                            float(point['z'])
                        ])
                    except Exception:
                        coords.extend([0.0, 0.0, 0.0])

                coords = coords[:63]
                coords += [0.0] * (63 - len(coords))

                frame_vector.extend(coords)

            sequence.append(frame_vector)

        if len(sequence) == 0:
            continue

        sequence = np.array(sequence, dtype=np.float32)

        if len(sequence) >= MAX_LEN:
            sequence = sequence[:MAX_LEN]
        else:
            pad = np.zeros((MAX_LEN - len(sequence), 126))
            sequence = np.vstack([sequence, pad])

        sequences.append(sequence)
        labels.append(label_map[attachment_id])
        attachment_ids_list.append(attachment_id)

print("Обработка завершена")
print(len(sequences))

Начинаю обработку JSON...


20000it [01:58, 168.23it/s]

Обработка завершена
19990


Преобразование в numpy arrays

In [ ]:
X = np.array(sequences, dtype=np.float32)
y = np.array(labels)

print(X.shape)
print(y.shape)

(19990, 64, 126)
(19990,)


In [ ]:
# Velocity features
# Для модели 2 (и возможно следующих)
delta_X = np.diff(
    X,
    axis=1,
    prepend=X[:, 0:1, :]
)

X = np.concatenate(
    [X, delta_X],
    axis=2
)

print(X.shape)

(19990, 64, 252)


Нормализация данных

In [ ]:
X_mean = X.mean(axis=(0, 1), keepdims=True)
X_std = X.std(axis=(0, 1), keepdims=True)

X = (X - X_mean) / (X_std + 1e-6)

split train/val/test

In [ ]:
X_train_val, X_test, y_train_val, y_test, ids_train_val, ids_test = train_test_split(
    X,
    y,
    attachment_ids_list,
    test_size=0.1,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
    X_train_val,
    y_train_val,
    ids_train_val,
    test_size=0.1,
    random_state=42,
    stratify=y_train_val
)

Создание Dataset

In [ ]:
class GestureDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_dataset = GestureDataset(
    X_train,
    y_train
)

val_dataset = GestureDataset(
    X_val,
    y_val
)

test_dataset = GestureDataset(
    X_test,
    y_test
)

Создание DataLoader

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    num_workers=2,
    pin_memory=True
)

Создание baseline LSTM-модели

In [ ]:
'''
class LSTMClassifier(nn.Module):

    def __init__(
        self,
        input_size=126,
        hidden_size=128,
        num_layers=1,
        num_classes=1000
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        hidden = hidden[-1]
        hidden = self.dropout(hidden)

        logits = self.fc(hidden)

        return logits
'''

'\nclass LSTMClassifier(nn.Module):\n\n    def __init__(\n        self,\n        input_size=126,\n        hidden_size=128,\n        num_layers=1,\n        num_classes=1000\n    ):\n\n        super().__init__()\n\n        self.lstm = nn.LSTM(\n            input_size=input_size,\n            hidden_size=hidden_size,\n            num_layers=num_layers,\n            batch_first=True\n        )\n\n        self.dropout = nn.Dropout(0.3)\n\n        self.fc = nn.Linear(hidden_size, num_classes)\n\n    def forward(self, x):\n\n        output, (hidden, cell) = self.lstm(x)\n\n        hidden = hidden[-1]\n        hidden = self.dropout(hidden)\n\n        logits = self.fc(hidden)\n\n        return logits\n'

In [ ]:
# Модель 3
'''
class LSTMClassifier(nn.Module):

    def __init__(
        self,
        input_size=252,
        hidden_size=128,
        num_classes=1000,
        dropout=0.3
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_size * 2,
            num_classes
        )

    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        pooled = output.mean(dim=1)

        pooled = self.dropout(pooled)

        logits = self.fc(pooled)

        return logits
'''

In [ ]:
# Модель 2

class LSTMClassifier(nn.Module):

    def __init__(
        self,
        input_size=252,
        hidden_size=256,
        num_layers=2,
        num_classes=1000,
        dropout=0.3
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        self.batch_norm = nn.BatchNorm1d(
            hidden_size * 2
        )

        self.dropout = nn.Dropout(dropout)

        self.fc1 = nn.Linear(
            hidden_size * 2,
            256
        )

        self.relu = nn.ReLU()

        self.fc2 = nn.Linear(
            256,
            num_classes
        )

    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        pooled = output.mean(dim=1)

        pooled = self.batch_norm(pooled)

        pooled = self.dropout(pooled)

        x = self.fc1(pooled)

        x = self.relu(x)

        x = self.dropout(x)

        logits = self.fc2(x)

        return logits


'\nclass LSTMClassifier(nn.Module):\n\n    def __init__(\n        self,\n        input_size=252,\n        hidden_size=256,\n        num_layers=2,\n        num_classes=1000,\n        dropout=0.3\n    ):\n\n        super().__init__()\n\n        self.lstm = nn.LSTM(\n            input_size=input_size,\n            hidden_size=hidden_size,\n            num_layers=num_layers,\n            batch_first=True,\n            dropout=dropout,\n            bidirectional=True\n        )\n\n        self.batch_norm = nn.BatchNorm1d(\n            hidden_size * 2\n        )\n\n        self.dropout = nn.Dropout(dropout)\n\n        self.fc1 = nn.Linear(\n            hidden_size * 2,\n            256\n        )\n\n        self.relu = nn.ReLU()\n\n        self.fc2 = nn.Linear(\n            256,\n            num_classes\n        )\n\n    def forward(self, x):\n\n        output, (hidden, cell) = self.lstm(x)\n\n        pooled = output.mean(dim=1)\n\n        pooled = self.batch_norm(pooled)\n\n        pooled =

Инициализация модели

In [ ]:
'''
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

num_classes = len(label_encoder.classes_)

model = LSTMClassifier(
    num_classes=num_classes
).to(device)
'''

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'cpu'
)

model = LSTMClassifier(
    input_size=252,
    num_classes=num_classes
).to(device)

print(device)

cuda


Функция потерь и оптимизатор

In [ ]:
'''
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)
'''

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

Обучение модели

In [ ]:
'''
EPOCHS = 50

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    model.eval()

    val_preds = []
    val_targets = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)

            logits = model(X_batch)

            preds = logits.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(y_batch.numpy())

    val_acc = accuracy_score(val_targets, val_preds)

    print(
        f"Epoch {epoch+1} | "
        f"Loss: {avg_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )
'''
EPOCHS = 100

best_val_acc = 0

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(logits, y_batch)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    model.eval()

    val_preds = []
    val_targets = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)

            logits = model(X_batch)

            preds = logits.argmax(dim=1)

            val_preds.extend(
                preds.cpu().numpy()
            )

            val_targets.extend(
                y_batch.numpy()
            )

    val_acc = accuracy_score(
        val_targets,
        val_preds
    )


    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            'best_model.pt'
        )

    print(
        f"Epoch {epoch+1} | "
        f"Loss: {avg_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

Epoch 1 | Loss: 6.5849 | Val Acc: 0.0117
Epoch 2 | Loss: 5.8201 | Val Acc: 0.0406
Epoch 3 | Loss: 5.3111 | Val Acc: 0.0650
Epoch 4 | Loss: 4.9127 | Val Acc: 0.0961
Epoch 5 | Loss: 4.5554 | Val Acc: 0.1161
Epoch 6 | Loss: 4.2291 | Val Acc: 0.1300
Epoch 7 | Loss: 3.9506 | Val Acc: 0.1589
Epoch 8 | Loss: 3.7019 | Val Acc: 0.1822
Epoch 9 | Loss: 3.4762 | Val Acc: 0.1922
Epoch 10 | Loss: 3.2946 | Val Acc: 0.2211
Epoch 11 | Loss: 3.0958 | Val Acc: 0.2222
Epoch 12 | Loss: 2.9400 | Val Acc: 0.2411
Epoch 13 | Loss: 2.7873 | Val Acc: 0.2417
Epoch 14 | Loss: 2.6474 | Val Acc: 0.2633
Epoch 15 | Loss: 2.5274 | Val Acc: 0.2672
Epoch 16 | Loss: 2.4120 | Val Acc: 0.2739
Epoch 17 | Loss: 2.3024 | Val Acc: 0.2656
Epoch 18 | Loss: 2.2143 | Val Acc: 0.2794
Epoch 19 | Loss: 2.1199 | Val Acc: 0.2861
Epoch 20 | Loss: 2.0303 | Val Acc: 0.2950
Epoch 21 | Loss: 1.9473 | Val Acc: 0.3128
Epoch 22 | Loss: 1.8695 | Val Acc: 0.2978
Epoch 23 | Loss: 1.7924 | Val Acc: 0.3211
Epoch 24 | Loss: 1.7200 | Val Acc: 0.3122
E

Оценка качества модели

In [ ]:
model.eval()

preds = []
targets = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        logits = model(X_batch)

        pred = logits.argmax(dim=1)

        preds.extend(
            pred.cpu().numpy()
        )

        targets.extend(
            y_batch.numpy()
        )

acc = accuracy_score(
    targets,
    preds
)

print("Accuracy:", acc)

Accuracy: 0.38219109554777386
